# 02_03 — гибридный Word2Vec: словарь + supervised

Цель эксперимента — объединить контекстные признаки обучаемого Word2Vec-классификатора с информацией из словарных определений и текстовых описаний семантических тегов. Предобученные векторы НКРЯ используются как фиксированный экстрактор признаков и **не дообучаются**.

Notebook самодостаточен: он задаёт пути, читает CoNLL-U, формирует канонический порядок классов, загружает словарь, строит три сопоставимых варианта и оценивает их на одном разбиении. Fiction используется только как validation; news, poetry, social и wiki — только как test.


## 1. Зависимости и воспроизводимость

Ячейка установки предназначена прежде всего для Colab. При локальном запуске зависимости берутся из `requirements.txt` проекта.


In [1]:
%pip install -q gensim scipy pandas scikit-learn tqdm trackio datasets natasha


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 38.0 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 18.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 70.7 MB/s eta 0:00:00:00:0100:01


In [2]:
# Для нового Colab-runtime раскомментируйте две строки ниже.
# !git clone https://github.com/tertiadecima/phd_semantics.git /content/phd_semantics
# %cd /content/phd_semantics


In [3]:
import ast
import hashlib
import importlib.util
import json
import os
import pickle
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
CONTEXT_WINDOW_DICTIONARY = 3
CONTEXT_WINDOW_SUPERVISED = 5
TOP_K = 3
SENTENCES_PER_CHECKPOINT = 2000
SGD_CONFIG = dict(
    loss='log_loss', penalty='elasticnet', alpha=1e-5, l1_ratio=0.05,
    class_weight='balanced', max_iter=50, early_stopping=False, tol=1e-3,
)

random.seed(SEED)
np.random.seed(SEED)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Корень git-репозитория не найден')


PROJECT_ROOT = find_project_root()
IS_COLAB = (importlib.util.find_spec('google') is not None and
            importlib.util.find_spec('google.colab') is not None)
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
else:
    DRIVE_ROOT = PROJECT_ROOT

DATA_ROOT = DRIVE_ROOT / 'data' if (DRIVE_ROOT / 'data').exists() else DRIVE_ROOT
MODELS_ROOT = DRIVE_ROOT / 'models' if (DRIVE_ROOT / 'models').exists() else DRIVE_ROOT
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
CACHE_ROOT = OUTPUT_ROOT / 'experiment_cache' / '02_03_word2vec_hybrid'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints' / '02_03_word2vec_hybrid'
TRACKIO_ROOT = OUTPUT_ROOT / 'trackio'
for directory in (CACHE_ROOT, CHECKPOINT_ROOT, TRACKIO_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['TRACKIO_DIR'] = str(TRACKIO_ROOT)

MODEL_CANDIDATES = [
    MODELS_ROOT / 'main_POS_V300_W5_C5/main_POS_V300_W5_C5.model',
    MODELS_ROOT / 'main_POS_V300_W5_C5.model',
    DRIVE_ROOT / 'main_POS_V300_W5_C5.model',
]
MODEL_PATH = next((path for path in MODEL_CANDIDATES if path.exists()), MODEL_CANDIDATES[0])
MODEL_FILES = [MODEL_PATH, MODEL_PATH.with_name(MODEL_PATH.name + '.wv.vectors.npy'),
               MODEL_PATH.with_name(MODEL_PATH.name + '.syn1neg.npy')]
missing = [path for path in MODEL_FILES if not path.exists()]
if missing:
    raise FileNotFoundError('Не найдены файлы Word2Vec:\n' + '\n'.join(map(str, missing)))

CONLLU_CANDIDATES = [DATA_ROOT / 'conllu', DRIVE_ROOT / 'conllu']
CONLLU_ROOT = next((path for path in CONLLU_CANDIDATES if path.exists()), CONLLU_CANDIDATES[0])
TRAIN_FILES = sorted([*CONLLU_ROOT.glob('bot/*.conllu'), *CONLLU_ROOT.glob('cobald/*.conllu'),
                      *CONLLU_ROOT.glob('syntagrus/*.conllu')])
HAND_ROOT = CONLLU_ROOT / 'handlabeled'
VALID_FILE = HAND_ROOT / 'GramEval2020-test-fiction.conllu'
TEST_FILES = {genre: HAND_ROOT / f'GramEval2020-test-{genre}.conllu'
              for genre in ('news', 'poetry', 'social', 'wiki')}
missing = [path for path in [VALID_FILE, *TEST_FILES.values()] if not path.exists()]
if not TRAIN_FILES:
    missing.append(CONLLU_ROOT)
if missing:
    raise FileNotFoundError('Не найдены корпуса:\n' + '\n'.join(map(str, missing)))

DATA_SPLITS = {
    'train': [str(path) for path in TRAIN_FILES],
    'validation': [str(VALID_FILE)],
    'test': {genre: str(path) for genre, path in TEST_FILES.items()},
}
print('Режим:', 'Colab + Google Drive' if IS_COLAB else 'локальный')
print('Модель:', MODEL_PATH)
print('Train-файлов:', len(TRAIN_FILES), '| validation:', VALID_FILE.name,
      '| test:', ', '.join(TEST_FILES))


FileNotFoundError: Корень git-репозитория не найден

## 2. Чтение CoNLL-U и единые метрики

Разметка `sem` нормализуется одинаково для всех корпусов. Многословные и пустые узлы CoNLL-U пропускаются. В итоговую выборку входят только токены, у которых поле `sem` действительно присутствует; пустой список тегов остаётся допустимой отрицательной разметкой.


In [ ]:
SEM_RE = re.compile(r"(?:^|\|)sem=(\[[^\]]*\]|[^|]+)")


def normalize_sem_tags(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_tags = value
    else:
        text = str(value).strip()
        try:
            parsed = ast.literal_eval(text)
            raw_tags = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            raw_tags = text.strip('[]').split(',')
    return sorted({str(tag).strip().strip("'\"").lower() for tag in raw_tags
                   if str(tag).strip().strip("'\"")})


def tags_from_misc(misc):
    match = SEM_RE.search(str(misc or ''))
    return (normalize_sem_tags(match.group(1)), True) if match else ([], False)


@dataclass
class Sentence:
    words: list[str]
    lemmas: list[str]
    upos: list[str]
    tags: list[list[str]]
    labeled: list[bool]
    source: str


def read_conllu(path):
    sentences, rows = [], []
    with Path(path).open(encoding='utf-8') as file:
        for line in file:
            line = line.rstrip('\n')
            if not line:
                if rows:
                    sentences.append(rows_to_sentence(rows, str(path)))
                    rows = []
                continue
            if line.startswith('#'):
                continue
            columns = line.split('\t')
            if len(columns) == 10 and columns[0].isdigit():
                rows.append(columns)
    if rows:
        sentences.append(rows_to_sentence(rows, str(path)))
    return [sentence for sentence in sentences if any(sentence.labeled)]


def rows_to_sentence(rows, source):
    tagged = [tags_from_misc(row[9]) for row in rows]
    return Sentence(
        words=[row[1] for row in rows],
        lemmas=[row[2] if row[2] != '_' else row[1].lower() for row in rows],
        upos=[row[3] for row in rows],
        tags=[value[0] for value in tagged],
        labeled=[value[1] for value in tagged],
        source=source,
    )


def file_signature(paths):
    digest = hashlib.sha256()
    for path in sorted(map(Path, paths)):
        stat = path.stat()
        digest.update(f'{path}:{stat.st_size}:{stat.st_mtime_ns}'.encode())
    return digest.hexdigest()[:16]


def atomic_json(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def atomic_pickle(value, path):
    # Pickle применяется только к checkpoint, созданному этим notebook в отдельной локальной папке.
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    temporary.replace(path)


def atomic_save_npy(array, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as file:
        np.save(file, array, allow_pickle=False)
        file.flush()
        os.fsync(file.fileno())
    temporary.replace(path)


def load_records(paths, name):
    records = []
    for path in tqdm(list(map(Path, paths)), desc=f'Чтение {name}', unit='файл'):
        records.extend(read_conllu(path))
    print(f'{name}: {len(records):,} размеченных предложений')
    return records


def tune_thresholds(y_true, probabilities, min_support=2):
    from sklearn.metrics import precision_recall_curve
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for column in range(y_true.shape[1]):
        if y_true[:, column].sum() < min_support:
            continue
        precision, recall, candidates = precision_recall_curve(y_true[:, column], probabilities[:, column])
        if not len(candidates):
            continue
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        thresholds[column] = candidates[int(np.nanargmax(f1))]
    return thresholds


def multilabel_metrics(y_true, probabilities, thresholds, classes, split, approach):
    from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
    predicted = (probabilities >= thresholds).astype(np.int8)
    micro = precision_recall_fscore_support(y_true, predicted, average='micro', zero_division=0)
    supported = y_true.sum(axis=0) > 0
    macro = precision_recall_fscore_support(y_true[:, supported], predicted[:, supported],
                                            average='macro', zero_division=0)
    per_class = precision_recall_fscore_support(y_true, predicted, average=None, zero_division=0)
    summary = {
        'approach': approach, 'split': split, 'tokens': int(len(y_true)), 'labels': int(y_true.sum()),
        'micro_precision': micro[0], 'micro_recall': micro[1], 'micro_f1': micro[2],
        'macro_precision_supported': macro[0], 'macro_recall_supported': macro[1],
        'macro_f1_supported': macro[2], 'exact_match': accuracy_score(y_true, predicted),
        'hamming_loss': hamming_loss(y_true, predicted),
    }
    details = pd.DataFrame({
        'approach': approach, 'split': split, 'tag': classes, 'precision': per_class[0],
        'recall': per_class[1], 'f1': per_class[2], 'support': per_class[3],
        'threshold': thresholds,
    }).sort_values('support', ascending=False)
    return summary, details


In [ ]:
train_records = load_records(TRAIN_FILES, 'train')
valid_records = load_records([VALID_FILE], 'validation_fiction')
test_records = {genre: load_records([path], f'test_{genre}') for genre, path in TEST_FILES.items()}

CLASSES = sorted({tag for records in [train_records, valid_records, *test_records.values()]
                  for sentence in records for tags, labeled in zip(sentence.tags, sentence.labeled)
                  if labeled for tag in tags})
if not CLASSES:
    raise ValueError('Не найдено ни одной метки sem')
TAG_TO_ID = {tag: index for index, tag in enumerate(CLASSES)}
atomic_json(CLASSES, CACHE_ROOT / 'canonical_classes.json')
atomic_json(DATA_SPLITS, CACHE_ROOT / 'data_splits.json')

counts = Counter(tag for sentence in train_records
                 for tags, labeled in zip(sentence.tags, sentence.labeled) if labeled for tag in tags)
display(pd.DataFrame({'tag': CLASSES, 'train_support': [counts[tag] for tag in CLASSES]})
        .sort_values('train_support', ascending=False).head(30))
print('Классов:', len(CLASSES))


## 3. Word2Vec, словарь и описания тегов

Обычные тексты словаря и описаний тегов токенизируются, лемматизируются и POS-размечаются Natasha, как в `02_01`. Ключ модели имеет формат `лемма_POS`; пунктуация, неизвестные POS и OOV-токены пропускаются.

Внешний pickle со словарными векторами не загружается. Для нужных в эксперименте лемм векторы строятся из исходного Hugging Face dataset, а затем сохраняются безопасно в `NPZ` (`allow_pickle=False`) и JSON. Word2Vec-модель и sklearn-checkpoints должны быть только из доверенной локальной папки проекта.


In [ ]:
from gensim.models import Word2Vec
from natasha import Doc, MorphVocab, NewsEmbedding, NewsMorphTagger, Segmenter

word2vec = Word2Vec.load(str(MODEL_PATH)).wv
POS_MAP = {
    'NOUN':'S', 'PROPN':'S', 'ADJ':'A', 'VERB':'V', 'AUX':'V', 'ADV':'ADV',
    'ADVPRO':'ADV', 'PRON':'SPRO', 'DET':'APRO', 'NUM':'NUM', 'ADP':'PR',
    'CCONJ':'CONJ', 'SCONJ':'CONJ', 'PART':'PART', 'INTJ':'INTJ',
}
POS_CLASSES = sorted({pos for records in [train_records, valid_records, *test_records.values()]
                      for sentence in records for pos in sentence.upos})
POS_TO_ID = {pos: index for index, pos in enumerate(POS_CLASSES)}

segmenter = Segmenter()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(NewsEmbedding())


def word_vector(lemma, upos):
    key = f'{lemma.lower()}_{POS_MAP.get(upos, upos)}'
    return word2vec[key] if key in word2vec else None


def text_vector(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    vectors = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        mapped_pos = POS_MAP.get(token.pos)
        if mapped_pos is None:
            continue
        key = f'{token.lemma.lower()}_{mapped_pos}'
        if key in word2vec:
            vectors.append(word2vec[key])
    return np.mean(vectors, axis=0).astype(np.float32) if vectors else None


TAG_DESCRIPTIONS = {
    'abstr':'абстрактные понятия, не имеющие физической формы', 'age':'возраст и возрастные характеристики',
    'animal':'животные и живые существа животного мира', 'appear':'появление, возникновение чего-либо',
    'artwork':'произведения искусства как объекты творчества', 'art':'искусство как деятельность и культурная сфера',
    'be':'существование, бытие, факт наличия', 'behav':'поведение и модели действий',
    'berry':'ягоды как биологический класс плодов', 'changest':'изменение состояния или характеристик',
    'cin':'кинематограф, кино как индустрия и искусство', 'class':'класс, категория или группа объектов',
    'cloth':'одежда и текстильные изделия', 'color':'цветовые характеристики и восприятие цвета',
    'concr':'конкретные предметные объекты', 'constr':'здания и строительные сооружения',
    'contact':'контактирование и взаимодействие через соприкосновение или связь',
    'degr':'степень, мера интенсивности или выраженности',
    'device':'высокотехнологичные устройства и автоматические системы', 'dir':'направление движения или ориентации',
    'disappear':'исчезновение или прекращение существования', 'disease':'болезни и патологические состояния',
    'dish':'посуда и предметы кухонного использования', 'dist':'расстояние между объектами',
    'doc':'документы и текстовые официальные материалы', 'dram':'театр и драматическое искусство',
    'emot':'эмоции и эмоциональные состояния', 'etn':'этнонимы и названия народов',
    'ev':'оценка, результат оценивания', 'famn':'фамилии как имена семейного происхождения',
    'food':'еда и пищевые продукты', 'form':'форма и структурные характеристики', 'fruit':'плоды растений',
    'furn':'мебель и предметы обстановки', 'grain':'зерновые культуры и зерно',
    'group':'группы людей или объединения', 'hum':'человек как биологический и социальный субъект',
    'humq':'качества человека и личностные характеристики', 'impact':'воздействие одного объекта на другой',
    'instr':'инструменты как средства выполнения действий', 'inter':'взаимодействие между объектами или субъектами',
    'kin':'родственные отношения', 'light':'свет и световые явления', 'max':'большая степень, максимум значения',
    'ment':'ментальная и когнитивная сфера', 'min':'малое значение, минимум',
    'move':'передвижение и движение', 'mus':'музыка как искусство и звуковая система',
    'neg':'негативная оценка или отрицательная характеристика',
    'org':'организации и институциональные структуры', 'part':'часть целого',
    'patrn':'отчество как часть имени', 'perc':'восприятие и перцептивные процессы',
    'physiol':'физиологические процессы живых организмов', 'physq':'физические свойства объектов',
    'place':'место и пространственные объекты', 'plant':'растения и ботанические объекты',
    'posit':'позитивная оценка или положительная характеристика', 'poss':'посессивная сфера, принадлежность',
    'prof':'профессии и виды профессиональной деятельности', 'propn':'имена собственные',
    'psych':'психика и психические процессы', 'put':'помещения и локализованные пространства',
    'qtm':'кванты и дискретные единицы', 'quant':'количество и количественные характеристики',
    'set':'множества и совокупности объектов', 'size':'размер и пространственные характеристики',
    'sound':'звук и акустические явления', 'speech':'речь и речевая деятельность',
    'speed':'скорость движения или изменения', 'stuff':'вещества и материалы',
    'taste':'вкус как сенсорная характеристика', 'temper':'температура и тепловые характеристики',
    'text':'текстовые объекты и письменные данные', 'time':'время и временные характеристики',
    'tool':'инструменты и орудия труда', 'topon':'топонимы и географические названия',
    'transp':'транспорт и средства передвижения', 'unit':'единицы измерения',
    'weapon':'оружие и военные средства', 'weather':'погодные явления', 'weight':'вес и масса объектов',
}
missing_descriptions = sorted(set(CLASSES) - set(TAG_DESCRIPTIONS))
if missing_descriptions:
    raise KeyError('Нет текстовых описаний классов: ' + ', '.join(missing_descriptions))

tag_vectors = []
for tag in CLASSES:
    vector = text_vector(TAG_DESCRIPTIONS[tag])
    if vector is None:
        raise ValueError(f'Описание тега {tag} не представимо в Word2Vec')
    tag_vectors.append(vector)
TAG_VECTORS = np.asarray(tag_vectors, dtype=np.float32)


def normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


TAG_VECTORS_NORM = normalize_rows(TAG_VECTORS)
print('Word2Vec:', word2vec.vector_size, '| POS:', len(POS_CLASSES), '| tag vectors:', TAG_VECTORS.shape)


In [ ]:
from datasets import load_dataset, load_from_disk

needed_lemmas = sorted({lemma.lower() for records in [train_records, valid_records, *test_records.values()]
                        for sentence in records for lemma, labeled in zip(sentence.lemmas, sentence.labeled)
                        if labeled})
lemma_signature = hashlib.sha256('\0'.join(needed_lemmas).encode()).hexdigest()[:16]
model_signature = file_signature(MODEL_FILES)
definition_stem = f'definitions_{lemma_signature}_{model_signature}'
DEFINITION_NPZ = CACHE_ROOT / f'{definition_stem}.npz'
DEFINITION_JSON = CACHE_ROOT / f'{definition_stem}.json'


def build_definition_cache():
    dataset_cache = DATA_ROOT / 'dictionary_dataset'
    if (dataset_cache / 'dataset_dict.json').exists():
        dictionary = load_from_disk(str(dataset_cache))
    else:
        dictionary = load_dataset('snagbreac/russian-reverse-dictionary-train-data',
                                  cache_dir=str(DATA_ROOT / 'huggingface_cache'))
        dictionary.save_to_disk(str(dataset_cache))
    needed = set(needed_lemmas)
    metadata, vectors = [], []
    for item in tqdm(dictionary['train'], desc='Векторизация нужных определений', unit='опред.'):
        lemma = item['word'].lower()
        if lemma not in needed:
            continue
        vector = text_vector(item['definition'])
        if vector is None:
            continue
        metadata.append({'lemma': lemma, 'definition': item['definition'], 'dict': item.get('df')})
        vectors.append(vector)
    matrix = np.asarray(vectors, dtype=np.float32)
    temporary = DEFINITION_NPZ.with_suffix('.tmp.npz')
    np.savez_compressed(temporary, vectors=matrix)
    temporary.replace(DEFINITION_NPZ)
    atomic_json({'model_signature': model_signature, 'lemma_signature': lemma_signature,
                 'items': metadata}, DEFINITION_JSON)


if not (DEFINITION_NPZ.exists() and DEFINITION_JSON.exists()):
    build_definition_cache()

definition_meta = json.loads(DEFINITION_JSON.read_text(encoding='utf-8'))
if (definition_meta.get('model_signature') != model_signature or
        definition_meta.get('lemma_signature') != lemma_signature):
    raise ValueError('Сигнатура кэша определений не совпадает с моделью или набором лемм')
with np.load(DEFINITION_NPZ, allow_pickle=False) as saved:
    definition_matrix = saved['vectors'].astype(np.float32)
if len(definition_meta['items']) != len(definition_matrix):
    raise ValueError('JSON и NPZ кэша определений имеют разную длину')

DEFINITIONS = {}
for item, vector in zip(definition_meta['items'], definition_matrix):
    DEFINITIONS.setdefault(item['lemma'], []).append({'definition': item['definition'], 'vector': vector})
print(f'Векторизовано определений: {len(definition_matrix):,}; лемм: {len(DEFINITIONS):,}/{len(needed_lemmas):,}')


## 4. Схема признаков

Для каждого размеченного токена строятся два набора.

**Supervised baseline (`02_02`)**: `[target_w2v(300), mean_context_±5(300), POS_one_hot]`. Целевой токен исключён из контекста.

**Hybrid**: `[target_w2v, mean_context_±3, POS_one_hot, coverage(3), topK_context_definition(K), max/mean_context_definition(2), context_tag(C), weighted_definition_tag(C), max_definition_tag(C)]`, где `C = len(CLASSES)`, `K = 3`. Определения ранжируются по косинусной близости к контексту; top-K дополняется значением `-1`. `weighted_definition_tag` — среднее сходство top-K определений с описанием каждого тега с softmax-весами контекстной близости, а `max_definition_tag` — максимум. Так несколько значений слова влияют на классификатор, но окончательное решение принимает обучаемая отдельная модель каждого тега.

При OOV целевого слова или пустом контексте используется нулевой вектор и отдельный coverage-флаг. При отсутствии векторизованных определений словарные признаки равны `-1`; пример не удаляется. Все строки поэтому имеют одну размерность и остаются сопоставимыми между подходами.


In [ ]:
def cosine_vector(vector, normalized_matrix):
    norm = float(np.linalg.norm(vector))
    return normalized_matrix @ (vector / max(norm, 1e-12))


def context_vector(vectors, index, window):
    neighbors = [vectors[j] for j in range(max(0, index-window), min(len(vectors), index+window+1))
                 if j != index and vectors[j] is not None]
    return np.mean(neighbors, axis=0).astype(np.float32) if neighbors else None


def dictionary_features(lemma, context, top_k=TOP_K):
    classes = len(CLASSES)
    definitions = DEFINITIONS.get(lemma.lower(), [])
    if not definitions:
        return (np.full(top_k + 2 + 3 * classes, -1.0, dtype=np.float32),
                np.full(classes, -1.0, dtype=np.float32), False)
    matrix = np.asarray([item['vector'] for item in definitions], dtype=np.float32)
    matrix_norm = normalize_rows(matrix)
    if context is None:
        context_sim = np.full(len(matrix), -1.0, dtype=np.float32)
        order = np.arange(min(top_k, len(matrix)))
        context_tag = np.full(classes, -1.0, dtype=np.float32)
    else:
        context_sim = cosine_vector(context, matrix_norm)
        order = np.argsort(context_sim)[::-1][:top_k]
        context_tag = cosine_vector(context, TAG_VECTORS_NORM).astype(np.float32)
    selected = matrix_norm[order]
    selected_context = context_sim[order]
    definition_tag = selected @ TAG_VECTORS_NORM.T
    if context is None:
        weights = np.full(len(order), 1.0 / len(order), dtype=np.float32)
    else:
        logits = (selected_context - selected_context.max()) / 0.1
        weights = np.exp(logits) / np.exp(logits).sum()
    weighted_tag = weights @ definition_tag
    max_tag = definition_tag.max(axis=0)
    padded = np.full(top_k, -1.0, dtype=np.float32)
    padded[:len(selected_context)] = selected_context
    aggregate = np.array([selected_context.max(), selected_context.mean()], dtype=np.float32)
    features = np.concatenate([padded, aggregate, context_tag, weighted_tag, max_tag]).astype(np.float32)
    # Словарный baseline 02_01 использует только самое близкое к контексту определение.
    baseline = definition_tag[0].astype(np.float32)
    return features, baseline, True


def build_features(records, show_progress=True):
    dimension = word2vec.vector_size
    supervised, hybrid, targets, dictionary_scores, statuses = [], [], [], [], []
    iterator = tqdm(records, desc='Гибридные признаки', unit='предл.') if show_progress else records
    for sentence in iterator:
        vectors = [word_vector(lemma, pos) for lemma, pos in zip(sentence.lemmas, sentence.upos)]
        for index, (tags, labeled) in enumerate(zip(sentence.tags, sentence.labeled)):
            if not labeled:
                continue
            target = vectors[index]
            context3 = context_vector(vectors, index, CONTEXT_WINDOW_DICTIONARY)
            context5 = context_vector(vectors, index, CONTEXT_WINDOW_SUPERVISED)
            target_vec = target if target is not None else np.zeros(dimension, dtype=np.float32)
            context3_vec = context3 if context3 is not None else np.zeros(dimension, dtype=np.float32)
            context5_vec = context5 if context5 is not None else np.zeros(dimension, dtype=np.float32)
            pos_vec = np.zeros(len(POS_CLASSES), dtype=np.float32)
            pos_vec[POS_TO_ID[sentence.upos[index]]] = 1.0
            dict_features, dict_baseline, has_definition = dictionary_features(sentence.lemmas[index], context3)
            coverage = np.array([target is not None, context3 is not None, has_definition], dtype=np.float32)
            supervised.append(np.concatenate([target_vec, context5_vec, pos_vec]))
            hybrid.append(np.concatenate([target_vec, context3_vec, pos_vec, coverage, dict_features]))
            dictionary_scores.append(dict_baseline)
            row = np.zeros(len(CLASSES), dtype=np.int8)
            for tag in tags:
                row[TAG_TO_ID[tag]] = 1
            targets.append(row)
            statuses.append((target is not None, context3 is not None, has_definition))
    arrays = tuple(np.asarray(value) for value in (supervised, hybrid, targets, dictionary_scores, statuses))
    for array in arrays[:4]:
        if not np.isfinite(array).all():
            raise ValueError('В признаках обнаружены NaN или infinity')
    return arrays


FEATURE_SIGNATURE = hashlib.sha256(json.dumps({
    'classes': CLASSES, 'model': model_signature, 'definitions': definition_stem,
    'top_k': TOP_K, 'dictionary_window': CONTEXT_WINDOW_DICTIONARY,
    'supervised_window': CONTEXT_WINDOW_SUPERVISED, 'pos': POS_CLASSES,
}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]


def cached_features(records, name, sentences_per_checkpoint=SENTENCES_PER_CHECKPOINT):
    source_signature = file_signature(sorted({sentence.source for sentence in records}))
    job_dir = CHECKPOINT_ROOT / 'features' / f'{name}_{FEATURE_SIGNATURE}_{source_signature}'
    job_dir.mkdir(parents=True, exist_ok=True)
    parts = [[] for _ in range(5)]
    for start in tqdm(range(0, len(records), sentences_per_checkpoint), desc=f'Пакеты {name}'):
        end = min(start + sentences_per_checkpoint, len(records))
        checkpoint = job_dir / f'{start:07d}_{end:07d}.npz'
        if not checkpoint.exists():
            values = build_features(records[start:end], show_progress=False)
            temporary = checkpoint.with_suffix('.tmp.npz')
            np.savez_compressed(temporary, supervised=values[0], hybrid=values[1], y=values[2],
                                dictionary=values[3], statuses=values[4])
            temporary.replace(checkpoint)
        with np.load(checkpoint, allow_pickle=False) as saved:
            values = [saved[key] for key in ('supervised', 'hybrid', 'y', 'dictionary', 'statuses')]
        for destination, value in zip(parts, values):
            destination.append(value)
    return tuple(np.concatenate(part) for part in parts)


x_train_supervised, x_train_hybrid, y_train, p_train_dictionary, status_train = cached_features(train_records, 'train')
x_valid_supervised, x_valid_hybrid, y_valid, p_valid_dictionary, status_valid = cached_features(valid_records, 'valid_fiction')
print('Supervised:', x_train_supervised.shape, '| Hybrid:', x_train_hybrid.shape, '| y:', y_train.shape)
assert x_train_hybrid.shape[1] == x_valid_hybrid.shape[1]
assert np.isfinite(x_train_hybrid).all() and np.isfinite(x_valid_hybrid).all()


## 5. Smoke-test и обучение One-vs-Rest

Для каждого тега обучается независимый `SGDClassifier(loss='log_loss')`. Если train-столбец константен, используется `DummyClassifier`. Состояние сохраняется после каждого класса; при возобновлении проверяются классы, размерность и полная сигнатура конфигурации.

Небольшая проверка до полного обучения подтверждает конечность и одинаковую размерность признаков, а также работу `fit`/`predict_proba`. Она не использует test.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier


def make_estimator(target, column):
    if np.unique(target).size == 1:
        return DummyClassifier(strategy='constant', constant=int(target[0]))
    return SGDClassifier(**SGD_CONFIG, random_state=SEED + column, n_jobs=1)


smoke_column = next((column for column in range(y_train.shape[1])
                     if np.unique(y_train[:min(2000, len(y_train)), column]).size == 2), None)
if smoke_column is not None:
    smoke_n = min(2000, len(y_train))
    smoke = make_estimator(y_train[:smoke_n, smoke_column], smoke_column)
    smoke.fit(x_train_hybrid[:smoke_n], y_train[:smoke_n, smoke_column])
    smoke_probability = smoke.predict_proba(x_valid_hybrid[:min(32, len(y_valid))])
    assert smoke_probability.shape[0] == min(32, len(y_valid)) and np.isfinite(smoke_probability).all()
    print('Smoke-test OK:', x_train_hybrid[:smoke_n].shape, '->', smoke_probability.shape)
else:
    print('Smoke-test классификатора пропущен: в первых 2000 строках нет вариативного класса')


class CheckpointedOVR:
    def __init__(self, estimators):
        self.estimators_ = estimators

    def predict_proba(self, x):
        columns = []
        for estimator in tqdm(self.estimators_, desc='Вероятности', unit='класс'):
            probabilities = estimator.predict_proba(x)
            if probabilities.shape[1] == 1:
                positive = np.full(len(x), float(estimator.classes_[0] == 1))
            else:
                positive = probabilities[:, int(np.where(estimator.classes_ == 1)[0][0])]
            columns.append(positive)
        return np.column_stack(columns)


RUN_CONFIG = {
    'seed': SEED, 'model_path': str(MODEL_PATH), 'model_signature': model_signature,
    'classes': CLASSES, 'train_sentences': len(train_records), 'validation_sentences': len(valid_records),
    'test_sentences': {name: len(records) for name, records in test_records.items()},
    'dictionary_window': CONTEXT_WINDOW_DICTIONARY, 'supervised_window': CONTEXT_WINDOW_SUPERVISED,
    'top_k': TOP_K, 'hybrid_feature_dim': int(x_train_hybrid.shape[1]),
    'supervised_feature_dim': int(x_train_supervised.shape[1]), 'classifier': SGD_CONFIG,
    'feature_signature': FEATURE_SIGNATURE,
}
RUN_SIGNATURE = hashlib.sha256(json.dumps(RUN_CONFIG, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_CONFIG, CHECKPOINT_ROOT / 'run_config.json')


def train_checkpointed_ovr(x, y, approach):
    model_file = CHECKPOINT_ROOT / f'{approach}_ovr_by_class.pkl'
    estimators = [None] * len(CLASSES)
    if model_file.exists():
        try:
            with model_file.open('rb') as file:
                saved = pickle.load(file)  # только доверенный локальный checkpoint этого notebook
            if (saved.get('classes') == CLASSES and saved.get('run_signature') == RUN_SIGNATURE and
                    saved.get('feature_dim') == x.shape[1]):
                estimators = saved['estimators']
        except (pickle.UnpicklingError, EOFError, AttributeError, ValueError, TypeError) as error:
            print(f'{approach}: checkpoint повреждён и будет пересоздан:', error)
    progress = tqdm(total=len(CLASSES), initial=sum(item is not None for item in estimators),
                    desc=f'Обучение {approach}', unit='класс')
    for column, tag in enumerate(CLASSES):
        if estimators[column] is not None:
            continue
        estimator = make_estimator(y[:, column], column)
        estimator.fit(x, y[:, column])
        estimators[column] = estimator
        atomic_pickle({'estimators': estimators, 'classes': CLASSES, 'run_signature': RUN_SIGNATURE,
                       'feature_dim': x.shape[1]}, model_file)
        progress.update(1)
        progress.set_postfix(tag=tag, positives=int(y[:, column].sum()))
    progress.close()
    return CheckpointedOVR(estimators)


import trackio
trackio.init(project='phd_semantics', name='02_03_word2vec_ruscorpora_hybrid',
             group='word2vec_hybrid', config=RUN_CONFIG)

supervised_classifier = train_checkpointed_ovr(x_train_supervised, y_train, 'supervised_baseline')
hybrid_classifier = train_checkpointed_ovr(x_train_hybrid, y_train, 'hybrid')

p_valid_supervised = supervised_classifier.predict_proba(x_valid_supervised)
p_valid_hybrid = hybrid_classifier.predict_proba(x_valid_hybrid)
THRESHOLDS = {
    'dictionary_baseline': tune_thresholds(y_valid, p_valid_dictionary),
    'supervised_baseline': tune_thresholds(y_valid, p_valid_supervised),
    'hybrid': tune_thresholds(y_valid, p_valid_hybrid),
}
for name, thresholds in THRESHOLDS.items():
    atomic_save_npy(thresholds, CHECKPOINT_ROOT / f'{name}_thresholds.npy')
atomic_json({'classes': CLASSES, 'run_signature': RUN_SIGNATURE,
             'threshold_files': {name: f'{name}_thresholds.npy' for name in THRESHOLDS}},
            CHECKPOINT_ROOT / 'thresholds_manifest.json')


## 6. Сравнение на одном разбиении

Порог каждого класса и каждого подхода выбирается **только на validation (fiction)** по максимальному F1. Test-жанры не участвуют в выборе признаков, параметров, порогов или остановке обучения.

Сводка содержит micro precision/recall/F1, macro-F1 по классам с поддержкой, exact match и Hamming loss. Отдельная таблица показывает precision/recall/F1 каждого класса. Coverage-отчёт показывает, сколько размеченных примеров имело OOV-цель, пустой вектор контекста или не имело векторизованного определения; такие строки сохранены с нулевыми векторами/флагами, а не исключены из оценки.


In [ ]:
def coverage_row(statuses, split):
    statuses = np.asarray(statuses, dtype=bool)
    total = len(statuses)
    return {
        'split': split, 'examples': total,
        'target_vector_missing': int((~statuses[:, 0]).sum()),
        'context_vector_missing': int((~statuses[:, 1]).sum()),
        'definition_missing': int((~statuses[:, 2]).sum()),
        'fully_represented': int(statuses.all(axis=1).sum()),
        'fully_represented_share': float(statuses.all(axis=1).mean()) if total else 0.0,
    }


summaries, details, coverage = [], [], [coverage_row(status_valid, 'validation_fiction')]
valid_probabilities = {
    'dictionary_baseline': p_valid_dictionary,
    'supervised_baseline': p_valid_supervised,
    'hybrid': p_valid_hybrid,
}
for approach, probabilities in valid_probabilities.items():
    summary, per_class = multilabel_metrics(y_valid, probabilities, THRESHOLDS[approach],
                                            CLASSES, 'validation_fiction', approach)
    summaries.append(summary)
    details.append(per_class)

for genre, records in test_records.items():
    x_supervised, x_hybrid, y_test, p_dictionary, statuses = cached_features(records, f'test_{genre}')
    coverage.append(coverage_row(statuses, f'test_{genre}'))
    probabilities_by_approach = {
        'dictionary_baseline': p_dictionary,
        'supervised_baseline': supervised_classifier.predict_proba(x_supervised),
        'hybrid': hybrid_classifier.predict_proba(x_hybrid),
    }
    for approach, probabilities in probabilities_by_approach.items():
        summary, per_class = multilabel_metrics(y_test, probabilities, THRESHOLDS[approach],
                                                CLASSES, f'test_{genre}', approach)
        summaries.append(summary)
        details.append(per_class)

summary_table = pd.DataFrame(summaries)
per_class_table = pd.concat(details, ignore_index=True)
coverage_table = pd.DataFrame(coverage)
display(summary_table)
display(coverage_table)
display(per_class_table.query("split == 'validation_fiction'").head(40))

summary_table.to_csv(CACHE_ROOT / 'comparison_metrics.csv', index=False)
per_class_table.to_csv(CACHE_ROOT / 'per_class_metrics.csv', index=False)
coverage_table.to_csv(CACHE_ROOT / 'coverage_report.csv', index=False)
atomic_json({'classes': CLASSES, 'run_signature': RUN_SIGNATURE, 'config': RUN_CONFIG},
            CACHE_ROOT / 'experiment_manifest.json')

trackio.log({
    f"{row['approach']}/{row['split']}/{metric}": float(row[metric])
    for row in summaries
    for metric in ('micro_precision', 'micro_recall', 'micro_f1',
                   'macro_f1_supported', 'exact_match', 'hamming_loss')
})
trackio.finish()

print('Сохранено:')
for path in sorted(CACHE_ROOT.glob('*')):
    print('-', path.relative_to(PROJECT_ROOT))
